# CTC Model Training Pipeline
This notebook implements the Connectionist Temporal Classification (CTC) pipeline. 
Unlike the sliding window approach, this trains the `CTC_CRNN` sequentially on entire audio recordings using PyTorch's native `CTCLoss`.

In [1]:
import sys

assert sys.version_info >= (3, 10)
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    !git clone https://github.com/stachuapa123/ASR_project.git
    %cd ASR_project
    # !git checkout <YOUR_BRANCH_NAME>  # Uncomment and set this to your branch if needed
    !pip install -q torchmetrics
    from google.colab import drive

    drive.mount("/content/drive")

    # Extract data securely if on Colab
    !mkdir -p "/content/asr_data"
    !unzip -q "/content/drive/MyDrive/asr_data.zip" -d "/content/asr_data"
    DATA_DIR = "/content/asr_data"
else:
    # Local path
    %load_ext autoreload
    %autoreload 2
    DATA_DIR = "../data"  # Update to your local subset or AutorskieDane

In [2]:
import torch
from torch.utils.data import DataLoader

from src.ctc.config import CTCConfig as C
from src.ctc.model import CTCModel
from src.ctc.dataset import CTCDataset, ctc_collate_fn
from src.ctc.augmentation import SpecAugment
from src.ctc.training import train_ctc, EarlyStopping

In [3]:
# Hyperparameters
PCT_VAL = 0.15
BATCH_SIZE = 16
N_EPOCHS = 100
LR = 1e-3
MAX_LR = 3e-3
WEIGHT_DECAY = 1e-4
PCT_START = 0.2
NUM_WORKERS = 4

device = C.get_device()
print(f"Using device: {device}")

Using device: cuda


In [4]:
dataset = CTCDataset(data_root=DATA_DIR, cache_mode=False, apply_augmentations=True)

n_total = len(dataset)
n_val = max(1, int(PCT_VAL * n_total))
n_train = n_total - n_val
generator = torch.Generator().manual_seed(42)
train_set, val_set = torch.utils.data.random_split(
    dataset,
    [n_train, n_val],
    generator=generator,
)
print(f"Train items: {len(train_set)} | Val items: {len(val_set)}")

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=ctc_collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

Train items: 8754 | Val items: 1544


In [5]:
model = CTCModel()
objective = torch.nn.CTCLoss(blank=C.BLANK_IDX, zero_infinity=True)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    steps_per_epoch=len(train_loader),
    epochs=N_EPOCHS,
    pct_start=PCT_START,
)
scaler = torch.amp.GradScaler(
    device=device.type,
    enabled=(device.type == "cuda"),
)
es = EarlyStopping()
spec_augment = SpecAugment()

In [6]:
model = train_ctc(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    objective=objective,
    device=device,
    n_epochs=N_EPOCHS,
    spec_augment=spec_augment,
    scheduler=scheduler,
    scaler=scaler,
    early_stopping=es,
    save_best_to="../trained_models/ctc_test.pt",
    use_amp=(device.type == "cuda"),
    step_scheduler_per_batch=True,
)

Epoch   1/100 | Train Loss: 3.3951 | Val Loss: 3.0233 | Val PER: 1.0000 | LR: 1.4e-04 [BEST]          
Epoch   2/100 | Train Loss: 2.5958 | Val Loss: 2.1222 | Val PER: 0.5730 | LR: 1.9e-04 [BEST]          
Epoch   3/100 | Train Loss: 2.0033 | Val Loss: 1.7169 | Val PER: 0.4566 | LR: 2.8e-04 [BEST]          
Epoch   4/100 | Train Loss: 1.7274 | Val Loss: 1.5442 | Val PER: 0.4236 | LR: 4.0e-04 [BEST]          
Epoch   5/100 | Train Loss: 1.5695 | Val Loss: 1.3657 | Val PER: 0.3672 | LR: 5.4e-04 [BEST]          
Epoch   6/100 | Train Loss: 1.4478 | Val Loss: 1.2627 | Val PER: 0.3409 | LR: 7.1e-04 [BEST]          
Epoch   7/100 | Train Loss: 1.3529 | Val Loss: 1.1940 | Val PER: 0.3267 | LR: 9.1e-04 [BEST]          
Epoch   8/100 | Train Loss: 1.2607 | Val Loss: 1.1237 | Val PER: 0.3075 | LR: 1.1e-03 [BEST]          
Epoch   9/100 | Train Loss: 1.1891 | Val Loss: 1.0593 | Val PER: 0.2894 | LR: 1.3e-03 [BEST]          
Epoch  10/100 | Train Loss: 1.1233 | Val Loss: 1.0142 | Val PER: 0.2785 |

KeyboardInterrupt: 